# 距離指数と主系列フィッティングで星団までの距離を求める

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coxless/astro-sandbox/blob/main/distance/distance_modulus.ipynb)

星の見かけの明るさ(見かけの等級 $m$)は、星そのものの明るさ(絶対等級 $M$、10 pc から見たときの等級)と距離 $d$ で決まります。

$$
m - M = 5 \log_{10}\left(\frac{d}{10\,\mathrm{pc}}\right) + A
$$

$m - M$ を **距離指数** と呼びます。$A$ は星間塵による減光(等級)です。
$M$ が分かっている天体(**標準光源**)があれば、$m$ を測るだけで距離が分かります。

$$
d\,[\mathrm{pc}] = 10^{(m - M - A + 5)/5}
$$

このノートブックでやること:

1. 視差がよく分かっている近傍の星で絶対等級を出し、HR 図(色−等級図)を描く
2. HR 図の **主系列** を「色が分かれば絶対等級が分かる」物差しとして読み取る
3. プレアデス星団の星を主系列に重ねて距離指数を求め(**主系列フィッティング**)、Gaia の視差による距離と比べる

視差が使えない遠い星団でも、色と見かけの等級さえ測れば距離が出せる、というのがこの方法の利点です。

## 使うデータ(すべてオープン)

- **Gaia DR3** (ESA): `gaiadr3.gaia_source` — 視差、固有運動、G 等級、BP−RP 色

## 0. 準備

In [ ]:
# Colab には astroquery が入っていないのでインストールする
!pip install -q astroquery

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astroquery.gaia import Gaia

Gaia.ROW_LIMIT = -1  # 行数制限なし(クエリ側で件数を絞る)


def distance_modulus(d_pc):
    # 距離 [pc] → 距離指数 m - M
    return 5.0 * np.log10(d_pc / 10.0)


def distance_from_modulus(mu):
    # 距離指数 m - M → 距離 [pc]
    return 10.0 ** ((mu + 5.0) / 5.0)

## 1. 距離指数の感覚をつかむ

太陽の絶対等級はおよそ $M_V = 4.83$ です。太陽をいろいろな距離に置いたら何等級に見えるかを計算してみます。

In [ ]:
M_SUN_V = 4.83
for d in [10, 100, 1_000, 10_000, 100_000]:
    mu = distance_modulus(d)
    print(f"{d:>7,} pc: m - M = {mu:5.1f}  → 太陽は {M_SUN_V + mu:5.1f} 等級に見える")

距離が 10 倍になるごとに距離指数は 5 増えます(明るさは 1/100 になる)。
肉眼の限界(約 6 等)を考えると、太陽のような星が肉眼で見えるのは 20 pc 程度までです。

## 2. 近傍の星で HR 図を描く

太陽から 50 pc 以内(視差 > 20 mas)で、視差の精度がよい($p/\sigma_p > 20$)星を取ってきます。
これだけ近ければ減光はほぼ無視でき、視差から距離を正確に出せるので、絶対等級が求まります。

$$
M_G = G - 5\log_{10}\left(\frac{d}{10\,\mathrm{pc}}\right) = G + 5\log_{10}(p\,[\mathrm{mas}]) - 10
$$

In [ ]:
query = '''
SELECT source_id, parallax, parallax_error, phot_g_mean_mag, bp_rp, ruwe
FROM gaiadr3.gaia_source
WHERE parallax > 20
  AND parallax_over_error > 20
  AND ruwe < 1.4
  AND bp_rp IS NOT NULL
  AND phot_g_mean_mag IS NOT NULL
'''
job = Gaia.launch_job_async(query)
near = job.get_results().to_pandas()
near.columns = near.columns.str.lower()
near["abs_g"] = near["phot_g_mean_mag"] + 5 * np.log10(near["parallax"]) - 10
print(len(near), "件")
near.head()

In [ ]:
plt.figure(figsize=(6, 7))
plt.scatter(near["bp_rp"], near["abs_g"], s=1, c="k", alpha=0.3)
plt.gca().invert_yaxis()  # 明るい星(等級が小さい)を上に
plt.xlabel("BP - RP [mag]")
plt.ylabel("M_G [mag]")
plt.title("HR diagram of stars within 50 pc")
plt.xlim(-0.5, 5)
plt.show()

左上から右下へ伸びる帯が **主系列**(水素を燃やしている星)です。
右上にぽつぽつあるのが巨星、左下の小さな塊が白色矮星です。

主系列の星は、色(表面温度)が決まると絶対等級がほぼ決まります。これを物差しにします。

## 3. 主系列の「尾根」をなめらかな曲線で表す

色 BP−RP の関数として主系列の絶対等級 $M_G(\mathrm{BP{-}RP})$ を多項式で当てはめます。
巨星や白色矮星を外すために、当てはめ → 曲線から離れた星を除外 → 再当てはめ、を繰り返します。

色の範囲は、近傍の星が十分あって主系列がはっきりしている 0.5〜2.5 mag に絞ります。

In [ ]:
COLOR_MIN, COLOR_MAX = 0.5, 2.5
POLY_DEG = 5

ms = near[(near["bp_rp"] > COLOR_MIN) & (near["bp_rp"] < COLOR_MAX)].copy()

# 最初の推定: 色のビンごとの中央値(巨星・白色矮星は少数なので中央値はあまり動かない)
edges = np.arange(COLOR_MIN, COLOR_MAX + 0.1, 0.1)
centers = 0.5 * (edges[1:] + edges[:-1])
medians = ms.groupby(pd.cut(ms["bp_rp"], edges), observed=False)["abs_g"].median().to_numpy()
ok = np.isfinite(medians)
coef = np.polyfit(centers[ok], medians[ok], POLY_DEG)

# 曲線から 0.75 mag 以上離れた星を外して当てはめ直す
for _ in range(5):
    resid = ms["abs_g"] - np.polyval(coef, ms["bp_rp"])
    keep = np.abs(resid) < 0.75
    coef = np.polyfit(ms.loc[keep, "bp_rp"], ms.loc[keep, "abs_g"], POLY_DEG)

resid = ms["abs_g"] - np.polyval(coef, ms["bp_rp"])
keep = np.abs(resid) < 0.75
print(f"当てはめに使った星: {keep.sum()} / {len(ms)}")
print(f"主系列まわりのばらつき(標準偏差): {resid[keep].std():.2f} mag")


def ms_abs_g(bp_rp):
    # 主系列の絶対等級 M_G を色 BP-RP から求める
    return np.polyval(coef, bp_rp)

In [ ]:
x = np.linspace(COLOR_MIN, COLOR_MAX, 200)
plt.figure(figsize=(6, 7))
plt.scatter(near["bp_rp"], near["abs_g"], s=1, c="lightgray")
plt.scatter(ms.loc[keep, "bp_rp"], ms.loc[keep, "abs_g"], s=1, c="k", alpha=0.3, label="used for fit")
plt.plot(x, ms_abs_g(x), c="tab:red", lw=2, label="main-sequence ridge")
plt.gca().invert_yaxis()
plt.xlabel("BP - RP [mag]")
plt.ylabel("M_G [mag]")
plt.xlim(-0.5, 5)
plt.legend()
plt.show()

主系列の帯には 0.3 mag ほどの幅があります。
金属量や年齢の違い、それに連星(2 つの星の光が重なって最大 0.75 mag 明るく見える)が原因です。

## 4. プレアデス星団の星を集める

プレアデス(すばる)は約 1 億年前に生まれた若い散開星団です。
星団の星は空の同じ方向を同じ速さで動いているので、**固有運動**(天球上での年あたりの動き)で選び出せます。
視差は「答え合わせ」に使いたいので、メンバーの選び出しには使いません。

- 中心: 赤経 56.75°, 赤緯 +24.12°
- 平均固有運動: $\mu_{\alpha*} \approx +20.0$, $\mu_\delta \approx -45.5$ mas/yr(Gaia Collaboration, Babusiaux et al. 2018)

In [ ]:
CLUSTER_RA, CLUSTER_DEC = 56.75, 24.12   # [deg]
SEARCH_RADIUS = 2.0                      # [deg]
PM_RA, PM_DEC = 20.0, -45.5              # [mas/yr]
PM_TOL = 3.0                             # 固有運動の許容範囲 [mas/yr]

query = f'''
SELECT source_id, ra, dec, parallax, parallax_error, pmra, pmdec,
       phot_g_mean_mag, bp_rp, ruwe
FROM gaiadr3.gaia_source
WHERE 1 = CONTAINS(POINT('ICRS', ra, dec),
                   CIRCLE('ICRS', {CLUSTER_RA}, {CLUSTER_DEC}, {SEARCH_RADIUS}))
  AND SQRT(POWER(pmra - ({PM_RA}), 2) + POWER(pmdec - ({PM_DEC}), 2)) < {PM_TOL}
  AND phot_g_mean_mag < 17
  AND ruwe < 1.4
  AND bp_rp IS NOT NULL
'''
job = Gaia.launch_job_async(query)
pleiades = job.get_results().to_pandas()
pleiades.columns = pleiades.columns.str.lower()
print(len(pleiades), "件")
pleiades.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

ax = axes[0]
ax.scatter(pleiades["pmra"], pleiades["pmdec"], s=3)
ax.set_xlabel("pmra [mas/yr]")
ax.set_ylabel("pmdec [mas/yr]")
ax.set_title("Proper motions of selected stars")
ax.set_aspect("equal")

ax = axes[1]
ax.scatter(pleiades["bp_rp"], pleiades["phot_g_mean_mag"], s=3, c="tab:blue")
ax.invert_yaxis()
ax.set_xlabel("BP - RP [mag]")
ax.set_ylabel("G [mag] (apparent)")
ax.set_title("Pleiades color-magnitude diagram")

plt.tight_layout()
plt.show()

見かけの等級で描いても、きれいな主系列が見えます。
これを上下にずらして近傍の主系列に重ねたとき、ずらした量が距離指数です。

## 5. 主系列フィッティング

星団の星それぞれについて、色から主系列の絶対等級 $M_G$ を読み取り、見かけの等級 $G$ との差(距離指数)を計算します。

ただし、プレアデスの方向には少しだけ星間塵があり、星が暗く・赤く見えます。
文献値 $E(B-V) \approx 0.04$ から、Gaia の波長帯でのおおよその値を使います。

- 色の赤化: $E(\mathrm{BP{-}RP}) \approx 1.3\,E(B-V) \approx 0.05$
- 減光: $A_G \approx 2.5\,E(B-V) \approx 0.10$

(正確な換算係数は星の色によって変わります。ここでは平均的な値です)

In [ ]:
E_BV = 0.04
E_BPRP = 1.3 * E_BV   # 色の赤化
A_G = 2.5 * E_BV      # G バンドの減光

pl = pleiades.copy()
pl["bp_rp0"] = pl["bp_rp"] - E_BPRP       # 減光を補正した本来の色
pl = pl[(pl["bp_rp0"] > COLOR_MIN) & (pl["bp_rp0"] < COLOR_MAX)].copy()
pl["mu"] = pl["phot_g_mean_mag"] - A_G - ms_abs_g(pl["bp_rp0"])

# 連星や非メンバーで外れた星の影響を減らすため、中央値を使い、外れ値を除いて繰り返す
mu = pl["mu"].median()
for _ in range(5):
    good = np.abs(pl["mu"] - mu) < 0.75
    mu = pl.loc[good, "mu"].median()

# 中央値の誤差の目安(ブートストラップ)
rng = np.random.default_rng(0)
vals = pl.loc[good, "mu"].to_numpy()
boot = [np.median(rng.choice(vals, len(vals))) for _ in range(1000)]
mu_err = np.std(boot)

d_ms = distance_from_modulus(mu)
print(f"使った星: {good.sum()} / {len(pl)}")
print(f"距離指数 m - M = {mu:.2f} ± {mu_err:.2f}")
print(f"主系列フィッティングによる距離: {d_ms:.0f} pc")

In [ ]:
x = np.linspace(COLOR_MIN, COLOR_MAX, 200)
plt.figure(figsize=(6, 7))
plt.scatter(pl["bp_rp0"], pl["phot_g_mean_mag"] - A_G, s=4, c="tab:blue", label="Pleiades (dereddened)")
plt.plot(x, ms_abs_g(x) + mu, c="tab:red", lw=2, label=f"field MS shifted by m-M = {mu:.2f}")
plt.gca().invert_yaxis()
plt.xlabel("(BP - RP)_0 [mag]")
plt.ylabel("G - A_G [mag]")
plt.legend()
plt.show()

## 6. 答え合わせ: Gaia の視差による距離

同じ星たちの視差から星団の距離を出して比べます。
1 つ 1 つの星の視差は誤差がありますが、たくさんの星で重み付き平均をとると精度がよくなります。

Gaia DR3 の視差には約 −0.017 mas のゼロ点オフセットがある(Lindegren et al. 2021)ので、補正してから使います。

In [ ]:
PLX_ZP = -0.017  # [mas]

plx = pl.loc[good, "parallax"] - PLX_ZP
w = 1.0 / pl.loc[good, "parallax_error"] ** 2
plx_mean = np.sum(w * plx) / np.sum(w)
plx_err = 1.0 / np.sqrt(np.sum(w))
d_plx = 1000.0 / plx_mean

print(f"平均視差: {plx_mean:.3f} ± {plx_err:.3f} mas")
print(f"視差による距離:           {d_plx:.1f} pc  (m - M = {distance_modulus(d_plx):.2f})")
print(f"主系列フィッティングの距離: {d_ms:.1f} pc  (m - M = {mu:.2f})")
print(f"差: {100 * (d_ms / d_plx - 1):+.1f} %")

文献ではプレアデスの距離は約 135 pc です。

主系列フィッティングの結果が視差の距離と数 % ずれるのは自然なことです。主な原因:

- **物差しの違い**: 物差しにした近傍の星は年齢も金属量もばらばら。プレアデスは若く、金属量も太陽に近い特定の値
- **減光の見積もり**: $A_G$ を 0.05 mag 間違えると距離は約 2% 変わる
- **連星**: 星団にも近傍の星にも連星が混じり、主系列が明るい側に引っぱられる

この「どれくらい合うか」の感覚が、視差が使えない遠い星団や、次の段(セファイド変光星など)の距離の信頼度を考えるときの基礎になります。

## 発展メモ

- `E_BV` や `PM_TOL`、`COLOR_MIN`/`COLOR_MAX` を変えて、結果がどれくらい動くか試す
- ヒアデス(赤経 66.75°, 赤緯 +15.87°、ただし非常に近く空に大きく広がる)やプレセペ(赤経 130.1°, 赤緯 +19.7°)など、ほかの星団でもやってみる

## 参考・クレジット

- Gaia Collaboration, Vallenari et al. (2023), *Gaia Data Release 3: Summary of the content and survey properties*, A&A 674, A1
- Gaia Collaboration, Babusiaux et al. (2018), *Gaia Data Release 2: Observational Hertzsprung-Russell diagrams*, A&A 616, A10
- Lindegren et al. (2021), *Gaia Early Data Release 3: Parallax bias versus magnitude, colour, and position*, A&A 649, A4

This work has made use of data from the European Space Agency (ESA) mission *Gaia* (https://www.cosmos.esa.int/gaia), processed by the Gaia Data Processing and Analysis Consortium (DPAC, https://www.cosmos.esa.int/web/gaia/dpac/consortium).